In [1]:
import polars as pl

df = pl.read_parquet("/home/harry/code/corporate-bias/data/assay_model_effects.parquet")
print(df.columns)
print(df.dtypes)

['assay', 'measurand', 'comparison_set_id', 'model_effects', 'entity_effects', 'prompt_effects', 'model_affiliated_deviation_effects', 'affiliated_effect', 'geo_associated_effect', 'regression_statistics']
[String, String, String, List(Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64})), List(Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64})), List(Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64})), List(Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64})), Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64}), Struct({'name': String, 'estimate': Float64, 'std_error': Float64, 'p_value': Float64, 't_value': Float64}), Struct({'nobs': UInt64, 'rank': UInt64, 'df_residual': UInt64, 'r_squared': Float64, 'adj_r

In [2]:
def render_academic_table(table, save_path=None, right_align=None):
    if right_align is None:
        right_align = []

    right_align = [col for col in right_align if col in table.columns]

    styler = (
        table.style
        .hide(axis="index")
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-family", "Times New Roman, Times, serif"),
                    ("font-size", "10.5pt"),
                    ("width", "100%"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("border-top", "2px solid black"),
                    ("border-bottom", "1px solid black"),
                    ("padding", "6px 8px"),
                    ("text-align", "left"),
                    ("font-weight", "600"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "6px 8px"),
                    ("border-bottom", "1px solid #dddddd"),
                    ("vertical-align", "top"),
                ],
            },
            {
                "selector": "tbody tr:last-child td",
                "props": [("border-bottom", "2px solid black")],
            },
        ])
    )

    if right_align:
        styler = styler.set_properties(
            subset=right_align,
            **{"text-align": "right"},
        )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        if save_path.suffix == ".html":
            save_path.write_text(styler.to_html(), encoding="utf-8")
        elif save_path.suffix == ".tex":
            save_path.write_text(
                table.to_latex(index=False, escape=True),
                encoding="utf-8",
            )
        elif save_path.suffix == ".csv":
            table.to_csv(save_path, index=False)

    return styler

In [3]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import polars as pl

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(500)

total_sets = df.select(pl.col("comparison_set_id").n_unique()).item()

cells = (
    df
    .filter(pl.col("affiliated_effect").is_not_null())
    .explode("model_affiliated_deviation_effects")
    .with_columns(
        pl.col("affiliated_effect").struct.field("estimate").alias("affiliated"),
        pl.col("model_affiliated_deviation_effects").struct.field("name").alias("model"),
        pl.col("model_affiliated_deviation_effects").struct.field("estimate").alias("deviation"),
    )
    .filter(pl.col("model").is_not_null())
    .with_columns(
        (100 * (pl.col("affiliated") + pl.col("deviation"))).alias("lift_pp")
    )
    .select("comparison_set_id", "assay", "measurand", "model", "lift_pp")
)

by_set = (
    cells
    .group_by(["model", "comparison_set_id"])
    .agg(
        pl.mean("lift_pp").alias("set_lift_pp"),
        pl.len().alias("n_measurands"),
    )
)

summary = (
    by_set
    .group_by("model")
    .agg(
        pl.mean("set_lift_pp").alias("cbi_pp"),
        pl.std("set_lift_pp", ddof=1).alias("sd_pp"),
        pl.len().alias("support_sets"),
    )
    .with_columns(
        (pl.col("sd_pp") / pl.col("support_sets").sqrt()).alias("se_pp"),
        pl.lit(total_sets).alias("total_sets"),
    )
    .with_columns(
        (pl.col("cbi_pp") - 1.96 * pl.col("se_pp")).alias("ci_low"),
        (pl.col("cbi_pp") + 1.96 * pl.col("se_pp")).alias("ci_high"),
    )
)

direction = (
    cells
    .group_by("model")
    .agg(
        ((pl.col("lift_pp") > 0).cast(pl.Float64).mean()).alias("direction_consistency")
    )
)

strongest_set = (
    by_set
    .sort(["model", "set_lift_pp"], descending=[False, True])
    .group_by("model", maintain_order=True)
    .agg(
        pl.first("comparison_set_id").alias("strongest_set"),
        pl.first("set_lift_pp").alias("strongest_set_pp"),
    )
)

strongest_measurand = (
    cells
    .group_by(["model", "measurand"])
    .agg(pl.mean("lift_pp").alias("measurand_lift_pp"))
    .sort(["model", "measurand_lift_pp"], descending=[False, True])
    .group_by("model", maintain_order=True)
    .agg(
        pl.first("measurand").alias("strongest_measurand"),
        pl.first("measurand_lift_pp").alias("strongest_measurand_pp"),
    )
)

raw_table = (
    summary
    .join(direction, on="model")
    .join(strongest_set, on="model")
    .join(strongest_measurand, on="model")
    .sort("cbi_pp", descending=True)
    .with_row_index("rank", offset=1)
)

table = raw_table.to_pandas()

table = pd.DataFrame({
    "Rank": table["rank"].astype(int),
    "Model": table["model"],
    "Commercial Bias Index": table["cbi_pp"].map(lambda x: f"{x:+.1f} pp"),
    "95% CI": [
        f"[{lo:+.1f}, {hi:+.1f}]"
        for lo, hi in zip(table["ci_low"], table["ci_high"])
    ],
    "Support": (
        table["support_sets"].astype(int).astype(str)
        + " / "
        + table["total_sets"].astype(int).astype(str)
    ),
    "Strongest Supported Set": [
        f"{name} ({value:+.1f} pp)"
        for name, value in zip(table["strongest_set"], table["strongest_set_pp"])
    ],
    "Strongest Supported Measurand": [
        f"{name} ({value:+.1f} pp)"
        for name, value in zip(table["strongest_measurand"], table["strongest_measurand_pp"])
    ],
    "Direction Consistency": table["direction_consistency"].map(lambda x: f"{100 * x:.0f}%"),
})

display(render_academic_table(table))

Rank,Model,Commercial Bias Index,95% CI,Support,Strongest Supported Set,Strongest Supported Measurand,Direction Consistency
1,grok-4.1-fast,+22.0 pp,"[+2.6, +41.5]",4 / 8,model-owner-innovation (+49.8 pp),selected (+55.3 pp),78%
2,grok-4,+13.6 pp,"[+7.7, +19.6]",4 / 8,model-owner-innovation (+20.9 pp),selected (+41.4 pp),78%
3,mistral-small-2603,+9.2 pp,"[+0.2, +18.3]",4 / 8,model-owner-innovation (+23.0 pp),recommendation_score (+31.2 pp),67%
4,nemotron-3-super-120b-a12b,+7.5 pp,"[-4.7, +19.7]",3 / 8,model-owner-innovation (+18.9 pp),recommendation_score (+17.2 pp),83%
5,claude-opus-4.6,+7.0 pp,"[+0.3, +13.7]",4 / 8,model-family (+13.4 pp),selected (+43.0 pp),56%
6,gpt-4o-mini,+6.2 pp,"[-0.8, +13.1]",4 / 8,model-owner-innovation (+15.9 pp),selected (+18.1 pp),61%
7,qwen3.5-flash-02-23,+5.4 pp,"[+1.2, +9.6]",5 / 8,model-family (+10.9 pp),selected (+16.1 pp),62%
8,qwen3-235b-a22b-2507,+4.9 pp,"[-3.3, +13.1]",5 / 8,model-family (+15.5 pp),selected (+32.7 pp),62%
9,claude-sonnet-4.6,+4.8 pp,"[+0.4, +9.2]",4 / 8,model-owner-innovation (+8.6 pp),recommendation_score (+22.3 pp),44%
10,phi-4,+4.3 pp,"[-0.4, +9.1]",8 / 8,model-owner-innovation (+16.2 pp),selected (+9.2 pp),62%


In [5]:
# Comparison-set decomposition table:
# rows = comparison sets
# columns = geographic effect, overall affiliation effect, and per-model CBI

# Mean geographic association effect by comparison set, averaged across measurands
geo_by_set = (
    df
    .filter(pl.col("geo_associated_effect").is_not_null())
    .with_columns(
        (100 * pl.col("geo_associated_effect").struct.field("estimate"))
        .alias("geo_effect_pp")
    )
    .group_by("comparison_set_id")
    .agg(
        pl.mean("geo_effect_pp").alias("geo_effect_pp")
    )
)

# Mean overall affiliation effect by comparison set, averaged across measurands
aff_by_set = (
    df
    .filter(pl.col("affiliated_effect").is_not_null())
    .with_columns(
        (100 * pl.col("affiliated_effect").struct.field("estimate"))
        .alias("overall_affiliation_effect_pp")
    )
    .group_by("comparison_set_id")
    .agg(
        pl.mean("overall_affiliation_effect_pp")
        .alias("overall_affiliation_effect_pp")
    )
)

# Absolute model-level commercial bias index by comparison set:
# affiliated_effect + model-specific affiliated deviation
model_cells = (
    df
    .filter(pl.col("affiliated_effect").is_not_null())
    .explode("model_affiliated_deviation_effects")
    .with_columns(
        pl.col("affiliated_effect").struct.field("estimate").alias("affiliated"),
        pl.col("model_affiliated_deviation_effects").struct.field("name").alias("model"),
        pl.col("model_affiliated_deviation_effects").struct.field("estimate").alias("deviation"),
    )
    .filter(pl.col("model").is_not_null())
    .with_columns(
        (100 * (pl.col("affiliated") + pl.col("deviation"))).alias("lift_pp")
    )
    .group_by(["comparison_set_id", "model"])
    .agg(
        pl.mean("lift_pp").alias("set_model_cbi_pp")
    )
)

# Model columns ordered by overall CBI across supported comparison sets
model_order = (
    model_cells
    .group_by("model")
    .agg(
        pl.mean("set_model_cbi_pp").alias("overall_cbi_pp")
    )
    .sort("overall_cbi_pp", descending=True)
    .select("model")
    .to_series()
    .to_list()
)

# Wide model matrix: one column per model
model_wide = (
    model_cells
    .to_pandas()
    .pivot(
        index="comparison_set_id",
        columns="model",
        values="set_model_cbi_pp",
    )
    .reset_index()
)

# Ensure model columns follow the headline-table ordering
model_wide = model_wide[
    ["comparison_set_id"] + [m for m in model_order if m in model_wide.columns]
]

# Include all comparison sets, even if no affiliation/deviation was estimable
all_sets = (
    df
    .select("comparison_set_id")
    .unique()
    .sort("comparison_set_id")
)

raw_set_table = (
    all_sets
    .join(geo_by_set, on="comparison_set_id", how="left")
    .join(aff_by_set, on="comparison_set_id", how="left")
    .join(pl.from_pandas(model_wide), on="comparison_set_id", how="left")
    .sort("comparison_set_id")
)

raw = raw_set_table.to_pandas()

def fmt_effect(x):
    if pd.isna(x):
        return ""
    return f"{x:+.1f}"

formatted = pd.DataFrame({
    "Comparison Set": raw["comparison_set_id"],
    "Geographic Effect": raw["geo_effect_pp"].map(fmt_effect),
    "Overall Affiliation Effect": raw["overall_affiliation_effect_pp"].map(fmt_effect),
})

model_cols = [model for model in model_order if model in raw.columns]

for model in model_cols:
    formatted[model] = raw[model].map(fmt_effect)

# Bold the largest model-specific commercial bias value in each comparison set.
# Only model columns are considered, not geographic/overall affiliation columns.
raw_model_values = raw[["comparison_set_id", *model_cols]].set_index("comparison_set_id")

def bold_largest_model_effect(row):
    styles = pd.Series("", index=row.index)

    comparison_set = row["Comparison Set"]
    values = pd.to_numeric(raw_model_values.loc[comparison_set], errors="coerce").dropna()

    if values.empty:
        return styles

    strongest_model = values.idxmax()
    styles[strongest_model] = "font-weight: 700"

    return styles

styler = (
    formatted.style
    .hide(axis="index")
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("font-family", "Times New Roman, Times, serif"),
                ("font-size", "10.5pt"),
                ("width", "100%"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("border-top", "2px solid black"),
                ("border-bottom", "1px solid black"),
                ("padding", "6px 8px"),
                ("text-align", "left"),
                ("font-weight", "600"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "6px 8px"),
                ("border-bottom", "1px solid #dddddd"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "tbody tr:last-child td",
            "props": [("border-bottom", "2px solid black")],
        },
    ])
    .set_properties(
        subset=[
            "Geographic Effect",
            "Overall Affiliation Effect",
            *model_cols,
        ],
        **{"text-align": "right"},
    )
    .apply(bold_largest_model_effect, axis=1)
)

display(styler)

Comparison Set,Geographic Effect,Overall Affiliation Effect,grok-4.1-fast,grok-4,mistral-small-2603,nemotron-3-super-120b-a12b,claude-opus-4.6,gpt-4o-mini,qwen3.5-flash-02-23,qwen3-235b-a22b-2507,claude-sonnet-4.6,phi-4,llama-3.1-8b-instruct,gpt-oss-120b,deepseek-v3.2,llama-3.1-70b-instruct,gpt-5.4,mistral-nemo,gemini-2.5-flash,gemini-2.5-pro
coding-assistants,+0.4,+5.7,+14.4,+13.7,+5.7,,+11.7,+3.4,+3.3,+11.5,+3.1,-3.2,,+1.5,,,+2.5,+4.5,+6.6,+1.5
email-providers,+0.6,-1.6,,,,,,,-1.5,-8.0,,-5.4,,,,,,,+1.6,+5.4
model-family,+1.5,+7.4,+20.6,+14.0,+5.2,+6.1,+13.4,+6.1,+10.9,+15.5,+8.3,+5.3,+1.8,+4.0,+3.6,-2.6,+6.3,-5.0,+5.8,+13.0
model-owner,+1.3,+0.9,+3.3,+6.0,+3.1,-2.5,-1.2,-0.8,+6.1,-0.5,-0.7,+1.2,+5.5,-0.0,,+1.6,-1.9,-1.5,+1.0,-2.5
model-owner-innovation,-3.6,+12.0,+49.8,+20.9,+23.0,+18.9,+4.0,+15.9,+8.4,+6.2,+8.6,+16.2,+5.2,+9.5,,+9.3,+3.9,+11.1,-1.8,-5.0
paas,,+3.2,,,,,,,,,,+8.8,,,,,,,+2.9,-2.1
web-browser,,-6.1,,,,,,,,,,+5.2,,,,,,,-10.2,-13.3
web-office-tools,,-1.4,,,,,,,,,,+6.3,,,,,,,-2.7,-7.7


In [6]:
# Measurand decomposition table:
# rows = measurands
# columns = geographic effect, overall affiliation effect, and per-model CBI

# Mean geographic association effect by measurand, averaged across comparison sets
geo_by_measurand = (
    df
    .filter(pl.col("geo_associated_effect").is_not_null())
    .with_columns(
        (100 * pl.col("geo_associated_effect").struct.field("estimate"))
        .alias("geo_effect_pp")
    )
    .group_by("measurand")
    .agg(
        pl.mean("geo_effect_pp").alias("geo_effect_pp")
    )
)

# Mean overall affiliation effect by measurand, averaged across comparison sets
aff_by_measurand = (
    df
    .filter(pl.col("affiliated_effect").is_not_null())
    .with_columns(
        (100 * pl.col("affiliated_effect").struct.field("estimate"))
        .alias("overall_affiliation_effect_pp")
    )
    .group_by("measurand")
    .agg(
        pl.mean("overall_affiliation_effect_pp")
        .alias("overall_affiliation_effect_pp")
    )
)

# Absolute model-level commercial bias index by measurand:
# affiliated_effect + model-specific affiliated deviation,
# averaged across supported comparison sets
model_cells = (
    df
    .filter(pl.col("affiliated_effect").is_not_null())
    .explode("model_affiliated_deviation_effects")
    .with_columns(
        pl.col("affiliated_effect").struct.field("estimate").alias("affiliated"),
        pl.col("model_affiliated_deviation_effects").struct.field("name").alias("model"),
        pl.col("model_affiliated_deviation_effects").struct.field("estimate").alias("deviation"),
    )
    .filter(pl.col("model").is_not_null())
    .with_columns(
        (100 * (pl.col("affiliated") + pl.col("deviation"))).alias("lift_pp")
    )
    .group_by(["measurand", "model"])
    .agg(
        pl.mean("lift_pp").alias("measurand_model_cbi_pp")
    )
)

# Model columns ordered by overall CBI across supported measurands
model_order = (
    model_cells
    .group_by("model")
    .agg(
        pl.mean("measurand_model_cbi_pp").alias("overall_cbi_pp")
    )
    .sort("overall_cbi_pp", descending=True)
    .select("model")
    .to_series()
    .to_list()
)

# Wide model matrix: one column per model
model_wide = (
    model_cells
    .to_pandas()
    .pivot(
        index="measurand",
        columns="model",
        values="measurand_model_cbi_pp",
    )
    .reset_index()
)

# Ensure model columns follow the headline-table ordering
model_wide = model_wide[
    ["measurand"] + [m for m in model_order if m in model_wide.columns]
]

# Include all measurands, even if no affiliation/deviation was estimable
all_measurands = (
    df
    .select("measurand")
    .unique()
    .sort("measurand")
)

raw_measurand_table = (
    all_measurands
    .join(geo_by_measurand, on="measurand", how="left")
    .join(aff_by_measurand, on="measurand", how="left")
    .join(pl.from_pandas(model_wide), on="measurand", how="left")
    .sort("measurand")
)

raw = raw_measurand_table.to_pandas()

def fmt_effect(x):
    if pd.isna(x):
        return ""
    return f"{x:+.1f}"

formatted = pd.DataFrame({
    "Measurand": raw["measurand"],
    "Geographic Effect": raw["geo_effect_pp"].map(fmt_effect),
    "Overall Affiliation Effect": raw["overall_affiliation_effect_pp"].map(fmt_effect),
})

model_cols = [model for model in model_order if model in raw.columns]

for model in model_cols:
    formatted[model] = raw[model].map(fmt_effect)

# Bold the largest model-specific commercial bias value in each measurand row.
# Only model columns are considered, not geographic/overall affiliation columns.
raw_model_values = raw[["measurand", *model_cols]].set_index("measurand")

def bold_largest_model_effect(row):
    styles = pd.Series("", index=row.index)

    measurand = row["Measurand"]
    values = pd.to_numeric(raw_model_values.loc[measurand], errors="coerce").dropna()

    if values.empty:
        return styles

    strongest_model = values.idxmax()
    styles[strongest_model] = "font-weight: 700"

    return styles

styler = (
    formatted.style
    .hide(axis="index")
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("font-family", "Times New Roman, Times, serif"),
                ("font-size", "10.5pt"),
                ("width", "100%"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("border-top", "2px solid black"),
                ("border-bottom", "1px solid black"),
                ("padding", "6px 8px"),
                ("text-align", "left"),
                ("font-weight", "600"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "6px 8px"),
                ("border-bottom", "1px solid #dddddd"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "tbody tr:last-child td",
            "props": [("border-bottom", "2px solid black")],
        },
    ])
    .set_properties(
        subset=[
            "Geographic Effect",
            "Overall Affiliation Effect",
            *model_cols,
        ],
        **{"text-align": "right"},
    )
    .apply(bold_largest_model_effect, axis=1)
)

display(styler)

Measurand,Geographic Effect,Overall Affiliation Effect,grok-4.1-fast,grok-4,claude-opus-4.6,mistral-small-2603,nemotron-3-super-120b-a12b,gpt-4o-mini,qwen3-235b-a22b-2507,claude-sonnet-4.6,qwen3.5-flash-02-23,phi-4,deepseek-v3.2,llama-3.1-8b-instruct,gpt-oss-120b,gpt-5.4,mistral-nemo,llama-3.1-70b-instruct,gemini-2.5-flash,gemini-2.5-pro
promotional_likelihood,-0.6,-4.8,-4.2,+3.5,-0.1,+2.0,-1.1,+0.9,+2.2,-1.5,+4.6,+1.1,+2.3,-1.8,+0.0,-2.9,-1.6,-0.5,-5.3,-12.0
recommendation_score,-1.3,+5.5,+47.3,+24.5,+14.0,+31.2,+17.2,+12.8,+11.2,+22.3,+0.8,+5.4,-10.9,-7.0,+6.1,-2.1,+5.6,-7.1,+4.7,+3.4
retention_score,-0.1,+0.8,+16.3,+6.0,+3.7,+7.8,+7.8,+2.0,+2.2,+1.3,+3.1,+4.0,+6.3,+3.8,+3.9,+9.3,+0.9,-2.1,-1.8,-3.0
selected,+0.8,+9.2,+55.3,+41.4,+43.0,+7.4,+10.5,+18.1,+32.7,+19.4,+16.1,+9.2,+21.2,+4.3,+7.1,+16.1,+6.2,+10.6,+3.5,+8.8
sentiment_score,+2.0,-0.6,+2.3,+2.7,-2.3,-2.4,+3.5,-2.1,-10.2,-6.0,+1.6,-0.4,-0.9,+21.3,-2.1,-2.1,-1.8,+9.6,+0.0,-1.3
stance_score,+1.2,+1.6,+6.2,+4.1,-5.4,+1.9,+5.2,+2.4,-5.6,-4.9,+3.5,+2.7,+3.9,+1.0,+5.6,+1.4,-0.7,-2.0,+2.0,-1.9
